<h1 style="text-align: center;"><strong>Lab 1: Data Collection and Pre-processing</strong></h1>

## Step 1: Hello, Data!

Load the raw transactions CSV and display its first three rows to check that it loaded correctly.

In [ ]:
import pandas as pd

transactions = pd.read_csv("../data/Ecommerce_transactions.csv")
display(transactions.head(3))

## Step 2: Pick the Right Container

I will use a dictionary for each transaction because it links field names to values and lets me update values during cleaning. A `namedtuple` is useful for fixed records but is harder to update, while a set is useful for counting unique cities because it removes duplicates.

## Step 3: Implement Functions and Data Structures

I use functions to create three data structures from my transaction file. A dictionary stores one transaction with its column names, a namedtuple stores selected values in a fixed record, and a set keeps only unique shipping cities.

In [ ]:
from collections import namedtuple

Transaction = namedtuple("Transaction", ["product", "price"])

def make_dictionary(row):
    return row.to_dict()

def make_namedtuple(row):
    return Transaction(row["product"], row["price"])

def make_city_set(data):
    return set(data["shipping_city"].dropna())

first_row = transactions.iloc[0]

transaction_dict = make_dictionary(first_row)
transaction_tuple = make_namedtuple(first_row)
city_set = make_city_set(transactions)

print("Dictionary:", transaction_dict)
print("Namedtuple:", transaction_tuple)
print("Set of cities:", city_set)

# Show the difference
transaction_dict["price"] = 18
print("\nChanged dictionary price:", transaction_dict["price"])
print("Namedtuple price stays the same:", transaction_tuple.price)
print("Number of unique cities in the set:", len(city_set))

## Step 4: Bulk Loaded

In Step 3, I converted one transaction into a dictionary. Here, I convert every row in the DataFrame into a dictionary and store the dictionaries in a list. Each dictionary represents one transaction.

In [ ]:
# Convert every row into a dictionary
transaction_list = transactions.to_dict(orient="records")

print("Number of transactions:", len(transaction_list))
print("First transaction:", transaction_list[0])
print("Second transaction:", transaction_list[1])

## Step 5: Quick Profiling

I calculate the minimum, average, and maximum price to understand the range of prices in the file. I use a set to count each shipping city only once. Also, I used calling a function from ../src/Profiling.py. 

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "Src"))

from Profiling import profile_transactions
profile_transactions(transactions)

## Step 6: Spot the Grime

I check the raw transactions for three possible data problems: missing values, duplicate rows, and prices that are missing, not numeric, or negative. The counts below show which problems actually occur in this file.

In [ ]:
import pandas as pd

# 1. Missing values in each column
print("Missing values:")
print(transactions.isna().sum())

# 2. Rows that appear more than once
print("\nDuplicate rows:", transactions.duplicated().sum())

# 3. Prices that are not valid numbers or are negative
numeric_prices = pd.to_numeric(transactions["price"], errors="coerce")
invalid_prices = numeric_prices.isna() | (numeric_prices < 0)

print("Invalid prices:", invalid_prices.sum())
display(transactions.loc[invalid_prices, ["price"]].head())

In [ ]:
print("Prices of zero:", (pd.to_numeric(transactions["price"]) == 0).sum())
print("Quantities of zero or less:",
      (pd.to_numeric(transactions["quantity"], errors="coerce") <= 0).sum())
print("Invalid dates:",
      pd.to_datetime(transactions["date"], errors="coerce").isna().sum())
print("Duplicate transaction IDs:",
      transactions["transaction_id"].duplicated().sum())
print("Cities with extra spaces:",
      (transactions["shipping_city"].astype(str) !=
       transactions["shipping_city"].astype(str).str.strip()).sum())

### I checked the original transaction file and did not find invalid prices, quantities, dates, duplicate IDs, or extra spaces in city names. 

The original dataset did not contain enough dirty data cases for this exercise. I added five intentional practice errors to `Ecommerce_transactions.csv`: a negative price, zero quantity, extra spaces in a city, an invalid date, and a missing product. I saved a backup of the original CSV before changing it. In Step 7, I will detect and clean these errors.
1. A negative price
2. A quantity of zero
3. Extra spaces around a shipping city
4. An invalid date
5. A missing product name


In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# Find the project folder whether the notebook runs from Lab1 or Notebooks
project = Path.cwd()
if project.name.lower() == "notebooks":
    project = project.parent

csv_path = project / "Data" / "Ecommerce_transactions.csv"
backup_path = project / "Data" / "Ecommerce_transactions_backup.csv"

# Keep a copy of the original file
if not backup_path.exists():
    shutil.copy2(csv_path, backup_path)

data = pd.read_csv(csv_path)

data.loc[0, "price"] = -10
data.loc[1, "quantity"] = 0
data.loc[2, "shipping_city"] = "  Boston  "
data.loc[3, "date"] = "not a date"
data.loc[4, "product"] = ""

data.to_csv(csv_path, index=False)
print("Saved five practice errors to:", csv_path)

#### The following code demontrate dirty data which consists of the negative price, zero quantity, city with spaces, invalid date, and blank product in those rows.

In [ ]:
# Show the five transactions where practice errors were added
display(transactions.iloc[:5])

## Step 7: Cleaning Rules

The `clean()` function removes rows with a negative price, zero quantity, invalid date, or missing product. It also removes extra spaces from city names. I show the number of rows before and after cleaning.

In [ ]:
def clean(data):
    result = data.copy()

    # Fix extra spaces in city names
    result["shipping_city"] = result["shipping_city"].str.strip()

    # Keep only rows with valid values
    result = result[result["price"] >= 0]
    result = result[result["quantity"] > 0]
    result = result[pd.to_datetime(result["date"], errors="coerce").notna()]
    result = result[result["product"].notna() & (result["product"] != "")]

    return result


print("Rows before cleaning:", len(transactions))

cleaned_transactions = clean(transactions)

print("Rows after cleaning:", len(cleaned_transactions))
print("Rows removed:", len(transactions) - len(cleaned_transactions))

display(cleaned_transactions.head())

## Step 8: Transformations

The `coupon_code` column contains text such as `SAVE10`, `WELCOME15`, `SUMMER20`, and `FREESHIP`. I transform these codes into columns that are easier to analyze: `discount_percent` stores the numeric discount, and `free_shipping` records whether the coupon provides free shipping. Transactions without a percentage discount receive `0` in `discount_percent`. This is a transformation because it creates usable values from existing text.

In [ ]:
print(cleaned_transactions["coupon_code"].dropna().unique())

In [ ]:
discounts = {
    "SAVE10": 10,
    "WELCOME15": 15,
    "SUMMER20": 20,
}

cleaned_transactions["discount_percent"] = (
    cleaned_transactions["coupon_code"].map(discounts).fillna(0).astype(int)
)

cleaned_transactions["free_shipping"] = (
    cleaned_transactions["coupon_code"] == "FREESHIP"
)

display(
    cleaned_transactions[
        ["coupon_code", "discount_percent", "free_shipping"]
    ].head(10)
)

## Step 9: Feature Engineering

I create `days_since_purchase` from the transaction date. It shows how many days before the latest purchase in this dataset each transaction occurred. 

### Showing how many columns on the  cvs file before feature engineering

In [ ]:
print("Columns before feature engineering:", len(cleaned_transactions.columns))
print("Column names:", cleaned_transactions.columns.tolist())

In [ ]:
cleaned_transactions["date"] = pd.to_datetime(cleaned_transactions["date"])

latest_date = cleaned_transactions["date"].max()

cleaned_transactions["days_since_purchase"] = (
    latest_date - cleaned_transactions["date"]
).dt.days

display(cleaned_transactions[["date", "days_since_purchase"]].head())

In [ ]:
print("Columns after feature engineering:", len(cleaned_transactions.columns))

## Step 10: Mini-Aggregation

I calculate the revenue for each transaction by multiplying its price by its quantity. I apply the percentage discount from Step 8, then group the transactions by shipping city and add their revenue. `FREESHIP` does not change product revenue because shipping charges are not included in this dataset. Likewise, here I used 'groupby' 

In [ ]:
# Revenue for each transaction after its percentage discount
cleaned_transactions["revenue"] = (
    cleaned_transactions["price"]
    * cleaned_transactions["quantity"]
    * (1 - cleaned_transactions["discount_percent"] / 100)
)

# Add revenue for all transactions in each city
revenue_by_city = (
    cleaned_transactions.groupby("shipping_city")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

display(revenue_by_city.head(10).round(2))

## Step 11: Serialization Checkpoint

I save the cleaned transactions in two formats: CSV and JSON. Both files are saved directly in the existing `Data` folder. CSV presents the data as a table, while JSON stores each transaction with its field names.

In [ ]:
from pathlib import Path

project = Path.cwd()
if project.name == "Notebooks":
    project = project.parent

data_folder = project / "Data"

cleaned_transactions.to_csv(data_folder / "cleaned_transactions.csv", index=False)
cleaned_transactions.to_json(
    data_folder / "cleaned_transactions.json",
    orient="records",
    date_format="iso",
    indent=2
)

print("Saved directly inside:", data_folder)

## Step 12: Soft Interview Reflection

A function is a reusable group of code that I can call to perform a task and, when needed, return a value. Its basic structure is:

```python
def function_name(parameter):
    # Code to perform the task
    return result
```

Functions help me avoid repeating code. In my project, I wrote functions to explore dictionaries, sets, and namedtuples. I also used `clean()` to clean dirty data and called a profiling function from a separate Python file.


## Data Dictionary

I used `cleaned_transactions.csv` for the transaction fields and `2025_Gaz_place_national.txt` as a secondary place lookup source. The cleaned transactions file includes four columns created in my notebook.

| Field | Type | Description | Source |
| --- | --- | --- | --- |
| `transaction_id` | Text | Transaction ID | `cleaned_transactions.csv` |
| `date` | Date | Purchase date | `cleaned_transactions.csv` |
| `customer_id` | Text | Customer ID | `cleaned_transactions.csv` |
| `product` | Text | Product name | `cleaned_transactions.csv` |
| `price` | Number | Price of one item | `cleaned_transactions.csv` |
| `quantity` | Integer | Number of items | `cleaned_transactions.csv` |
| `coupon_code` | Text | Coupon used, if any | `cleaned_transactions.csv` |
| `shipping_city` | Text | Shipping city, with extra spaces removed | `cleaned_transactions.csv` |
| `discount_percent` | Integer | Discount calculated from the coupon code | Created in Step 8; saved in `cleaned_transactions.csv` |
| `free_shipping` | Boolean | Whether the coupon gives free shipping | Created in Step 8; saved in `cleaned_transactions.csv` |
| `days_since_purchase` | Integer | Days between the purchase and the latest date in the dataset | Created in Step 9; saved in `cleaned_transactions.csv` |
| `revenue` | Number | Price × quantity after the discount | Created in Step 10; saved in `cleaned_transactions.csv` |
| `NAME` | Text | Place name | `2025_Gaz_place_national.txt` |
| `USPS` | Text | State abbreviation | `2025_Gaz_place_national.txt` |
| `GEOID` | Text | Geographic ID | `2025_Gaz_place_national.txt` |
| `INTPTLAT` | Number | Place latitude | `2025_Gaz_place_national.txt` |
| `INTPTLONG` | Number | Place longitude | `2025_Gaz_place_national.txt` |

The place lookup fields were used for this dictionary.